# ComfyUI on Google Colab (starter)

This notebook installs the official ComfyUI repository in the current Colab runtime and opens a temporary Cloudflare Quick Tunnel to it.

**Before running:** in Colab, select `Runtime → Change runtime type → T4 GPU` if a GPU is available. Free GPU availability is not guaranteed.

**Security warning:** the temporary tunnel URL acts like a shared access link. Anyone who gets it may reach your ComfyUI session. Do not use this notebook with sensitive/private images, do not publish the URL, and stop the tunnel when finished. This is a convenience starter, not a production-secure deployment.

In [ ]:
# Check the runtime before installing anything
import os, platform, sys
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch check failed:', repr(exc))


## Install ComfyUI

The notebook clones the official ComfyUI repository and installs its Python requirements. It does not download model weights or third-party custom nodes.

In [ ]:
from pathlib import Path
import subprocess, sys

COMFY_DIR = Path('/content/ComfyUI')
if not COMFY_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)
    ], check=True)
else:
    print('ComfyUI folder already exists; reusing it.')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-r',
    str(COMFY_DIR / 'requirements.txt')
], check=True)
print('ComfyUI installation step complete.')


## Start ComfyUI and the temporary tunnel

Keep this cell running while using ComfyUI. Copy the newly generated `trycloudflare.com` URL into another browser tab on your iPhone. Do not share it.

In [ ]:
from pathlib import Path
from queue import Queue, Empty
from threading import Thread
from urllib.request import urlopen
import re, subprocess, time

COMFY_DIR = Path('/content/ComfyUI')
COMFY_LOG = open('/content/comfyui.log', 'w')
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
    cwd=str(COMFY_DIR), stdout=COMFY_LOG, stderr=subprocess.STDOUT
)

ready = False
for _ in range(120):
    if comfy.poll() is not None:
        raise RuntimeError('ComfyUI stopped unexpectedly. Check /content/comfyui.log')
    try:
        with urlopen('http://127.0.0.1:8188/', timeout=2) as response:
            if response.status < 500:
                ready = True
                break
    except Exception:
        time.sleep(1)
if not ready:
    raise TimeoutError('ComfyUI did not become ready. Check /content/comfyui.log')
print('ComfyUI is responding on port 8188.')

cloudflared_path = '/usr/local/bin/cloudflared'
subprocess.run([
    'bash', '-lc',
    'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
    '-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared'
], check=True)

tunnel = subprocess.Popen(
    [cloudflared_path, 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
tunnel_lines = Queue()
def collect_tunnel_output():
    for output_line in tunnel.stdout:
        tunnel_lines.put(output_line)
Thread(target=collect_tunnel_output, daemon=True).start()

public_url = None
for _ in range(120):
    try:
        output_line = tunnel_lines.get(timeout=1)
    except Empty:
        if tunnel.poll() is not None:
            raise RuntimeError('Cloudflare tunnel exited unexpectedly.')
        continue
    print(output_line.rstrip())
    match = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', output_line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print('\nOPEN THIS URL IN YOUR IPHONE BROWSER:')
    print(public_url)
else:
    print('Tunnel URL not detected yet. Check the output above; the tunnel process may still be running.')


## Stop the services

Run this cell when you finish. Stopping the tunnel matters because its URL can provide access to the running ComfyUI instance.

In [ ]:
for process_name in ('tunnel', 'comfy'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except Exception:
            process.kill()
        print(f'Stopped {process_name}.')
if 'COMFY_LOG' in globals() and not COMFY_LOG.closed:
    COMFY_LOG.close()
print('Session services stopped. Colab may still retain temporary files until the runtime is deleted.')
